# Modelling

**Goal.** Predict hard drive failure from daily SMART readings, and choose a model that can be used in practice: scored by drive, with its alarm level set by cost.

Decisions carried over from the EDA (`01_data_loading_and_eda`):
- **Window:** 14 consecutive days, no missing day or empty reading, for failing and healthy windows alike.
- **Inputs:** SMART 5, 187, 188, 197 (core) and 4, 12, 190 (extra). `log1p` on 5, 187 and 197; 188 as a nonzero flag.
- **Dropped:** copies 198, 194 and 195; age columns 9 and 240.
- **Split:** by date, never random. Q2 2026 is not loaded here and is opened once at the very end.

In [7]:
import numpy as np
import pandas as pd

import gc, psutil
import numpy as np
import pandas as pd

def mem(): print(round(psutil.Process().memory_info().rss / 1e9, 2), "GB used by this notebook")

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_curve


In [8]:

# ---------- SETTINGS: change numbers here, nowhere else ----------
SEED = 42
SEEDS = [0, 1, 2, 3, 4]            # for repeating training runs

# What one example is
WINDOW_DAYS = 14                   # days of readings the model sees
HORIZON_DAYS = 14                  # label is 1 if the drive fails within this many days AFTER the window

# Date split (proposed, not confirmed)
TRAIN_START = pd.Timestamp("2023-01-01")
TRAIN_END   = pd.Timestamp("2025-12-31")
VAL_START   = pd.Timestamp("2026-01-01")
VAL_END     = pd.Timestamp("2026-03-31")
TEST_START  = pd.Timestamp("2026-04-01")
TEST_END    = pd.Timestamp("2026-06-30")

# An example on day t looks at failures up to day t + HORIZON_DAYS.
# So the last usable example day of each period stops HORIZON_DAYS before the next period starts.
GAP = pd.Timedelta(days=HORIZON_DAYS)
TRAIN_LAST_EXAMPLE = VAL_START - GAP - pd.Timedelta(days=1)
VAL_LAST_EXAMPLE   = TEST_START - GAP - pd.Timedelta(days=1)
TEST_LAST_EXAMPLE  = TEST_END - GAP      # labels must be known inside the quarter

# Inputs
CORE = ["smart_5_raw", "smart_187_raw", "smart_188_raw", "smart_197_raw"]
CANDIDATES = ["smart_4_raw", "smart_12_raw", "smart_190_raw"]

# Cost and scoring
COST_RATIOS = [5, 10, 20, 50, 100]   # cost of a missed failure / cost of one false alarm
TOP_K = [5, 10, 25]                  # "riskiest K drives today" watch-list sizes
N_BOOTSTRAP = 1000                   # resamples for the score ranges

# ---------- sanity checks ----------
assert TRAIN_END < VAL_START <= VAL_END < TEST_START <= TEST_END
print("last train example:", TRAIN_LAST_EXAMPLE.date())
print("last validation example:", VAL_LAST_EXAMPLE.date())
print("last test example:", TEST_LAST_EXAMPLE.date())

last train example: 2025-12-17
last validation example: 2026-03-17
last test example: 2026-06-16


In [9]:
# names the loading and window cells use
EXTRA = CANDIDATES
FEATURES = CORE + CANDIDATES
LATER_QUARTERS = ["Q1_2026"]      # loaded now; Q2_2026 stays unloaded until the final test
N_NEG_TRAIN = 1_000_000           # healthy windows kept for training
DATA_ROOT = "../FailurePrediction"        # folder holding clean_train.parquet and the Dataset/ folder
DATA_DIR = f"{DATA_ROOT}/Dataset"         # folder with the quarterly parquet files

In [ ]:
cols = ["serial_number", "date", "failure"] + FEATURES

train = pd.read_parquet(f"{DATA_ROOT}/clean_train.parquet", columns=cols)
train["serial_number"] = train["serial_number"].astype("category")

later = []
for q in LATER_QUARTERS:
    d = pd.read_parquet(f"{DATA_DIR}/seagate_data_{q}.parquet", columns=cols)
    d["date"] = pd.to_datetime(d["date"])
    nums = d.select_dtypes("float64").columns
    d[nums] = d[nums].astype("float32")
    later.append(d)
later = pd.concat(later, ignore_index=True)

# one shared list of drive IDs, then join
cats = pd.Index(train["serial_number"].cat.categories).union(pd.Index(later["serial_number"].unique()))
train["serial_number"] = train["serial_number"].cat.set_categories(cats)
later["serial_number"] = pd.Categorical(later["serial_number"], categories=cats)
df = pd.concat([train, later], ignore_index=True)
del train, later; _ = gc.collect()

# same cleaning rule as the EDA: each drive is cut at its first failure
df = df.sort_values(["serial_number", "date"]).reset_index(drop=True)
fail_day = df["date"].where(df["failure"] == 1)
first_fail = fail_day.groupby(df["serial_number"], observed=True).transform("min")
df = df.loc[first_fail.isna() | (df["date"] <= first_fail)].reset_index(drop=True)
del fail_day, first_fail; _ = gc.collect()

print(df.shape, "| drives:", df["serial_number"].nunique(), "| failures:", int(df["failure"].sum()))
print("dates:", df["date"].min().date(), "to", df["date"].max().date())
print(df.groupby(df["date"].dt.to_period("Q"))["failure"].sum().to_string())
mem()

## 1. Label design

**Question.** What is one example, and when is it labelled as failing?

**Method.**
- One example is one drive on one day (its end day). The input is that drive's last 14 days of readings, up to and including the end day, for 7 columns: SMART 5, 187, 188, 197 (core) and 4, 12, 190 (extra).
- The label is 1 if the drive fails 1 to 14 days after the end day, otherwise 0. The failure day itself never ends a window, so no input comes from the failure day or later.
- Every window needs 14 consecutive days and no empty reading, for failing and healthy windows alike.
- Healthy windows include the early days of drives that fail later, not only drives that never fail, because a data centre sees both kinds.
- All failing windows are kept. Healthy windows are sampled for training, because there are too many to keep. Validation keeps every window, so its failure share and false-alarm rate are real.
- Split by date: training windows end by 2025-12-17 and validation windows end between 2026-01-01 and 2026-03-17. The last 14 days of each period are dropped, so no label looks past the end of its period.

**Finding.** The training period gives 1,018,804 windows: 18,804 failing and 1,000,000 healthy, sampled from 19,946,452 available (each kept healthy window stands for about 20). 1,477 of the 1,560 failed drives (94.7%) have at least one failing window. Validation (Q1 2026) keeps every window: 1,381,980 windows, of which 1,101 are failing (0.080%, about 1 in 1,250), from 104 of the 129 drives that failed in the quarter. Failing windows come in runs of up to 14 per drive, so they are not independent. All leakage checks passed.

**Decision.** Keep this design. The training share (1.8%) is set by sampling; the validation share (0.080%) is the real one and is used for every score. Score ranges are computed by resampling drives, not windows. Window 14 days and horizon 14 days are starting values kept as settings; 7 and 21 are compared later.

In [ ]:
codes = df["serial_number"].cat.codes.to_numpy()
dates = df["date"].to_numpy()
n, W, H = len(df), WINDOW_DAYS, HORIZON_DAYS

# row number within each drive (rows are contiguous and in date order)
starts = np.r_[0, np.flatnonzero(np.diff(codes) != 0) + 1]
pos = np.arange(n) - np.repeat(starts, np.diff(np.r_[starts, n]))

# days from each row to its drive's failure (a huge number if the drive never fails)
fail_idx = np.flatnonzero(df["failure"].to_numpy() == 1)
fail_date = np.full(int(codes.max()) + 1, np.datetime64("2100-01-01"), dtype=dates.dtype)
fail_date[codes[fail_idx]] = dates[fail_idx]
days_to_fail = ((fail_date[codes] - dates) / np.timedelta64(1, "D")).astype(np.int32)

# rows that can END a complete window: W consecutive days, no missing reading
X = df[FEATURES].to_numpy(np.float32)
cum = np.r_[0, np.cumsum(np.isnan(X).any(axis=1))]
first_row = np.where(pos >= W - 1, np.arange(n) - (W - 1), 0)
span = ((dates - dates[first_row]) / np.timedelta64(1, "D")).astype(np.int32)
complete = (pos >= W - 1) & (span == W - 1) & ((cum[1:] - cum[first_row]) == 0)
del cum, first_row, span; _ = gc.collect()

label = ((days_to_fail >= 1) & (days_to_fail <= H)).astype(np.int8)
usable = complete & (days_to_fail >= 1)          # the failure day itself never ends a window

def in_period(first_example, last_example):      # last_example keeps the look-ahead inside the period
    return usable & (dates >= first_example.to_datetime64()) & (dates <= last_example.to_datetime64())

rng = np.random.default_rng(SEED)
tr_mask = in_period(TRAIN_START, TRAIN_LAST_EXAMPLE)
pos_idx = np.flatnonzero(tr_mask & (label == 1))
neg_all = np.flatnonzero(tr_mask & (label == 0))
neg_idx = rng.choice(neg_all, size=min(N_NEG_TRAIN, len(neg_all)), replace=False)
train_idx = np.sort(np.r_[pos_idx, neg_idx])
val_idx = np.flatnonzero(in_period(VAL_START, VAL_LAST_EXAMPLE))

def summarize(name, idx):
    y = label[idx]
    print(f"{name:6s} windows: {len(idx):>10,} | positives: {int(y.sum()):>6,} | positive share: {y.mean() * 100:6.3f}% | drives: {len(np.unique(codes[idx])):,}")

summarize("train", train_idx); summarize("val", val_idx)
print(f"\nhealthy training windows available: {len(neg_all):,} (kept {len(neg_idx):,}; each kept one stands for {len(neg_all) / len(neg_idx):.1f})")

fi = fail_idx[dates[fail_idx] <= TRAIN_END.to_datetime64()]
print(f"failed drives in the training period: {len(fi):,} | with at least one positive window: {int(np.isin(codes[fi], codes[pos_idx]).sum()):,}")
fv = fail_idx[(dates[fail_idx] >= VAL_START.to_datetime64()) & (dates[fail_idx] <= VAL_END.to_datetime64())]
print(f"failed drives in validation: {len(fv):,} | with at least one positive window: {int(np.isin(codes[fv], codes[val_idx][label[val_idx] == 1]).sum()):,}")

assert (days_to_fail[train_idx] >= 1).all() and (days_to_fail[val_idx] >= 1).all()   # no window ends on or after the failure day
assert dates[train_idx].max() <= TRAIN_LAST_EXAMPLE.to_datetime64()                  # train labels only use train-period failures
assert dates[val_idx].max() <= VAL_LAST_EXAMPLE.to_datetime64()                      # validation labels stay inside validation
assert dates[val_idx].min() > dates[train_idx].max()                                 # validation is later than training
print("leakage checks passed")
mem()

train  windows:  1,018,804 | positives: 18,804 | positive share:  1.846% | drives: 20,203
val    windows:  1,381,980 | positives:  1,101 | positive share:  0.080% | drives: 18,616

healthy training windows available: 19,946,452 (kept 1,000,000; each kept one stands for 19.9)
failed drives in the training period: 1,560 | with at least one positive window: 1,477
failed drives in validation: 129 | with at least one positive window: 104
leakage checks passed
3.93 GB used by this notebook


## 2. Model features

**Question.** What summary of each 14-day window should the model see?

**Method.** The model gets 20 numbers per window instead of the whole sequence, taken from the EDA:
- For counters 5, 187 and 197: the end value and the change across the window (both on a `log1p` scale), plus "nonzero at the end" and "rose inside the window" flags.
- For counter 188: only the two flags (the log did not help, EDA Section 4).
- For 4 and 12: the end value (log) and whether it changed.
- For 190 (temperature): the end value and the change.

In [ ]:
names = [c.replace("smart_", "").replace("_raw", "") for c in FEATURES]
J = {nm: i for i, nm in enumerate(names)}

def make_features(idx):
    first, last = X[idx - (W - 1)], X[idx]          # first and last day of each window
    f = {}
    for nm in ["5", "187", "197"]:
        a = np.log1p(np.clip(first[:, J[nm]], 0, None))
        b = np.log1p(np.clip(last[:, J[nm]], 0, None))
        f[f"{nm}_end_log"], f[f"{nm}_change_log"] = b, b - a
        f[f"{nm}_end_nz"], f[f"{nm}_rose"] = last[:, J[nm]] > 0, last[:, J[nm]] > first[:, J[nm]]
    f["188_end_nz"], f["188_rose"] = last[:, J["188"]] > 0, last[:, J["188"]] > first[:, J["188"]]
    for nm in ["4", "12"]:
        f[f"{nm}_end_log"] = np.log1p(np.clip(last[:, J[nm]], 0, None))
        f[f"{nm}_changed"] = last[:, J[nm]] != first[:, J[nm]]
    f["190_end"], f["190_change"] = last[:, J["190"]], last[:, J["190"]] - first[:, J["190"]]
    return pd.DataFrame(f).astype("float32")

F_train, F_val = make_features(train_idx), make_features(val_idx)
y_train, y_val = label[train_idx], label[val_idx]
print("features:", F_train.shape[1], "| train:", F_train.shape, "| val:", F_val.shape,
      "| NaNs:", int(F_train.isna().sum().sum() + F_val.isna().sum().sum()))

features: 20 | train: (1018804, 20) | val: (1381980, 20) | NaNs: 0


## 3. Models

**Question.** How well can simple models predict failure within 14 days, scored on validation data with the real failure share?

**Method.** Three models, fitted on the training windows and scored on every validation window:
- a rule: the number of counters (5, 187, 188, 197) that are nonzero at the end of the window;
- logistic regression on scaled features (scaler fitted on training data only);
- gradient boosting with shallow trees.

Failures are weighted up in training because they are rare. Scores: average precision (area under the precision-recall curve; a random model scores about the failure share) and the share of failing windows caught when only 1%, 0.1% or 0.01% of healthy windows raise an alarm. At 0.1% false alarms, about 1 healthy drive in 1,000 is flagged each day.

In [ ]:

def recall_at_fpr(yt, s, target):
    fpr, tpr, _ = roc_curve(yt, s)
    return float(tpr[fpr <= target].max())

sc = StandardScaler().fit(F_train)                       # scaler fitted on training data only
w_pos = (y_train == 0).sum() / (y_train == 1).sum()

scores = {}
scores["rule: counters nonzero"] = F_val[["5_end_nz", "187_end_nz", "188_end_nz", "197_end_nz"]].sum(axis=1).to_numpy()
lr = LogisticRegression(max_iter=2000, class_weight="balanced").fit(sc.transform(F_train), y_train)
scores["logistic"] = lr.predict_proba(sc.transform(F_val))[:, 1]
gb = HistGradientBoostingClassifier(max_depth=3, learning_rate=0.1, max_iter=200, random_state=SEED)
gb.fit(F_train, y_train, sample_weight=np.where(y_train == 1, w_pos, 1.0))
scores["grad_boost"] = gb.predict_proba(F_val)[:, 1]

rows = []
for k, s in scores.items():
    r = {"model": k, "avg_precision": average_precision_score(y_val, s)}
    for t in (0.01, 0.001, 0.0001):
        r[f"recall@{t:.2%}"] = recall_at_fpr(y_val, s, t)
    rows.append(r)
print("validation: windows", len(y_val), "| failing", int(y_val.sum()), "| failure share", round(y_val.mean() * 100, 3), "%")
print(pd.DataFrame(rows).round(3).to_string(index=False))
mem()

validation: windows 1381980 | failing 1101 | failure share 0.08 %
                 model  avg_precision  recall@1.00%  recall@0.10%  recall@0.01%
rule: counters nonzero          0.046         0.285         0.000         0.000
              logistic          0.129         0.672         0.264         0.067
            grad_boost          0.121         0.723         0.265         0.059
4.28 GB used by this notebook


## 4. Drive-level scores and ranges

**Question.** Of the 129 drives that fail in Q1 2026, how many does each model catch at least once, and how much could that number change with different data?

**Method.**
- A failed drive counts as caught if any of its failing windows raises an alarm.
- The alarm level is set so that only 1%, 0.1% or 0.01% of healthy windows raise one.
- Ranges come from resampling drives 1,000 times, not windows, because one drive gives up to 14 near-identical windows.
- 25 of the 129 failed drives have no failing window (14-day gap, missing days), so the best possible drive-level recall is 80.6%.

**Finding.** Of the 129 drives that fail in Q1 2026, the best possible drive-level recall is 80.6% (104 drives have at least one failing window). At 1% false alarms (10 per 1,000 drive-days), logistic regression catches 65% of failed drives (range 56-73%) and gradient boosting 67% (58-75%). At 0.1% false alarms: 36% (29-45%) and 38% (29-46%). At 0.01%: 9% and 12%. The two models' ranges overlap almost completely at every level.

**Decision.** Use logistic regression: it is no worse than gradient boosting, it is simpler, and each score can be explained reading by reading.

In [ ]:
# ---------- Drive-level scores with ranges (resampling DRIVES, not windows) ----------
cv = codes[val_idx]
all_d = np.union1d(cv, codes[fv])               # every validation drive, incl. failed drives with no window
di = np.searchsorted(all_d, cv)
nd = len(all_d)
fail_drive = np.isin(all_d, codes[fv])           # the 129 drives that fail in validation
is_pos, is_neg = (y_val == 1), (y_val == 0)
pos_per = np.bincount(di[is_pos], minlength=nd)
neg_per = np.bincount(di[is_neg], minlength=nd)

rng_b = np.random.default_rng(SEED)
boot_w = [np.bincount(rng_b.integers(0, nd, nd), minlength=nd) for _ in range(N_BOOTSTRAP)]

rows = []
for name in ["logistic", "grad_boost"]:
    s = scores[name]
    for t in (0.01, 0.001, 0.0001):
        thr = np.quantile(s[is_neg], 1 - t)      # alarm when score is above this
        alarm = s > thr
        pos_alarm = np.bincount(di[is_pos & alarm], minlength=nd)
        neg_alarm = np.bincount(di[is_neg & alarm], minlength=nd)
        caught = fail_drive & (pos_alarm > 0)    # failed drive with at least one alarmed failing window

        def rng95(f):
            v = np.array([f(w) for w in boot_w]); return np.percentile(v, [2.5, 97.5])
        d_pt = caught.sum() / fail_drive.sum()
        d_lo, d_hi = rng95(lambda w: (w * caught).sum() / (w * fail_drive).sum())
        w_pt = pos_alarm.sum() / pos_per.sum()
        w_lo, w_hi = rng95(lambda w: (w * pos_alarm).sum() / (w * pos_per).sum())
        rows.append({"model": name, "false alarms": f"{t:.2%}",
                     "per 1,000 drive-days": round(t * 1000, 1),
                     "drive recall": f"{d_pt:.3f} ({d_lo:.2f}-{d_hi:.2f})",
                     "window recall": f"{w_pt:.3f} ({w_lo:.2f}-{w_hi:.2f})"})

print(f"failed validation drives: {int(fail_drive.sum())} | best possible drive recall: "
      f"{int((fail_drive & (pos_per > 0)).sum()) / int(fail_drive.sum()):.3f}")
print(pd.DataFrame(rows).to_string(index=False))
mem()

NameError: name 'codes' is not defined

**Finding.** Of the 129 drives that fail in Q1 2026, the best possible drive-level recall is 80.6% (104 drives have at least one failing window). At 1% false alarms (10 per 1,000 drive-days), logistic regression catches 65% of failed drives (range 56-73%) and gradient boosting 67% (58-75%). At 0.1% false alarms: 36% (29-45%) and 38% (29-46%). At 0.01%: 9% and 12%. The two models' ranges overlap almost completely at every level.

**Decision.** The CNN was compared with this baseline in 02b_cnn_experiments.ipynb and was not shown to be better, so logistic regression is the model used.

## 5. What can be caught at all

**Question.** How many of the 129 failed validation drives could any model catch from these inputs?

**Method.**
- A failed drive with no failing window (14-day gap, missing days) cannot be scored.
- A failed drive is "quiet" if all four core counters (5, 187, 188, 197) are zero at the end of every one of its failing windows.
- The remaining drives are catchable. Logistic regression is checked at the 1% false-alarm level.

In [ ]:
s_log = scores["logistic"]
quiet_win = (F_val[["5_end_nz", "187_end_nz", "188_end_nz", "197_end_nz"]].sum(axis=1).to_numpy() == 0)
thr = np.quantile(s_log[is_neg], 1 - 0.01)
pa = np.bincount(di[is_pos & (s_log > thr)], minlength=nd)
caught = fail_drive & (pa > 0)
has_window = pos_per > 0
loud = np.bincount(di[is_pos & ~quiet_win], minlength=nd)
quiet_drive = fail_drive & has_window & (loud == 0)
no_window = fail_drive & ~has_window
catchable = fail_drive & has_window & ~quiet_drive
print("failed drives:", int(fail_drive.sum()))
print("  no failing window:", int(no_window.sum()), "| quiet:", int(quiet_drive.sum()), "| catchable:", int(catchable.sum()))
print(f"ceiling: {catchable.sum() / fail_drive.sum():.3f} | logistic catches {int(caught.sum())} at 1% false alarms ({caught.sum() / fail_drive.sum():.3f})")

failed drives: 129
  no failing window: 25 | quiet: 12 | catchable: 92
ceiling: 0.713 | logistic catches 84 at 1% false alarms (0.651)


## 6. Alarm level by cost

**Question.** At what alarm level should the model operate if a missed failure costs R times one false alarm?

**Method.**
- Cost = false alarms + R × missed failed drives on the validation quarter. A false alarm is one healthy drive-day flagged; failed drives with no window count as missed.
- For each R in `COST_RATIOS`, 61 alarm levels are tried (never alarm, then 0.001% to 5% false alarms) and the cheapest is kept. The score threshold is chosen on validation only and frozen before the Q2 test.
- Limits: consecutive-day alarms on one drive are counted each day; choosing and reporting on the same quarter is optimistic.

In [ ]:
n_val_days = len(np.unique(dates[val_idx])); n_fail = int(fail_drive.sum())
grid = np.r_[0.0, np.logspace(-5, -1.3, 60)]
curve = []
for t in grid:
    if t == 0:
        curve.append((0.0, np.inf, 0, 0)); continue
    thr = np.quantile(s_log[is_neg], 1 - t); alarm = s_log > thr
    pa = np.bincount(di[is_pos & alarm], minlength=nd)
    curve.append((float(t), float(thr), int(alarm[is_neg].sum()), int((fail_drive & (pa > 0)).sum())))

CHOSEN, rows = {}, []
for R in COST_RATIOS:
    costs = [fa + R * (n_fail - c) for (t, thr, fa, c) in curve]
    k = int(np.argmin(costs)); t, thr, fa, c = curve[k]
    CHOSEN[R] = {"false_alarm_rate": t, "threshold": thr}
    rows.append({"R": R, "false-alarm rate": f"{t:.3%}", "false alarms per day": round(fa / n_val_days, 1),
                 "failed drives caught": f"{c} of {n_fail}", "cost": int(costs[k]), "cost with no alarms": R * n_fail})
print(pd.DataFrame(rows).to_string(index=False))

  R false-alarm rate  false alarms per day failed drives caught  cost  cost with no alarms
  5           0.001%                   0.2             3 of 129   644                  645
 10           0.002%                   0.3             5 of 129  1265                 1290
 20           0.010%                   1.8            12 of 129  2480                 2580
 50           0.102%                  18.5            48 of 129  5453                 6450
100           0.102%                  18.5            48 of 129  9503                12900


## 7. Files for the app

**Question.** What does a small app need to score drives with this model?

**Method.**
- The fitted scaler and logistic regression, the 20 feature names, the window and horizon, and the alarm level for each cost ratio R (Section 6).
- The cost table (R, false-alarm rate, false alarms per day, failed drives caught) and a validation summary with the numbers for the app's limits panel.
- A small demo set of 20 drives taken from validation: 6 that fail (the window ending closest to 7 days before failure) and 14 that never fail, with their scores computed here so the app can be checked against them.

In [ ]:
import joblib, json, os
os.makedirs("model", exist_ok=True)

joblib.dump({"model": lr, "scaler": sc, "features": list(F_train.columns),
             "window_days": int(W), "horizon_days": int(H), "alarm_levels": CHOSEN},
            "model/baseline_logistic.joblib")
pd.DataFrame(rows).to_csv("model/cost_table.csv", index=False)

summary = {"validation_quarter": "2026-Q1", "failed_drives": n_fail, "no_window": int(no_window.sum()),
           "quiet": int(quiet_drive.sum()), "catchable": int(catchable.sum()), "drive_recall": {}}
for t in (0.01, 0.001, 0.0001):
    thr_ = np.quantile(s_log[is_neg], 1 - t)
    pa_ = np.bincount(di[is_pos & (s_log > thr_)], minlength=nd)
    c_ = fail_drive & (pa_ > 0)
    lo, hi = np.percentile([(w * c_).sum() / (w * fail_drive).sum() for w in boot_w], [2.5, 97.5])
    summary["drive_recall"][f"{t:.2%}"] = {"recall": float(c_.sum() / n_fail), "low": float(lo), "high": float(hi)}
json.dump(summary, open("model/validation_summary.json", "w"), indent=2)

# ---- demo drives (raw readings, Backblaze-style columns) ----
rng_demo = np.random.default_rng(SEED)
vcodes, vlab = codes[val_idx], label[val_idx]
demo_rows, demo_key = [], []

def add_window(end_row, name, failed):
    for k in range(W):
        r = end_row - (W - 1) + k
        demo_rows.append({"serial_number": name, "date": pd.Timestamp(dates[r]).date(),
                          **{c: float(X[r, j]) for j, c in enumerate(FEATURES)}})
    demo_key.append({"serial_number": name, "fails_within_14_days": failed,
                     "offline_score": float(s_log[np.searchsorted(val_idx, end_row)])})

n_f = 0
for code in rng_demo.permutation(np.unique(codes[fv])):
    rows_c = val_idx[(vcodes == code) & (vlab == 1)]
    if len(rows_c) == 0:
        continue
    add_window(rows_c[np.argmin(np.abs(days_to_fail[rows_c] - 7))], f"demo_fail_{n_f + 1:02d}", True)
    n_f += 1
    if n_f == 6:
        break
healthy_pool = val_idx[(vlab == 0) & (days_to_fail[val_idx] > 10_000)]
for i, end_row in enumerate(rng_demo.choice(healthy_pool, 14, replace=False)):
    add_window(end_row, f"demo_ok_{i + 1:02d}", False)

pd.DataFrame(demo_rows).to_csv("model/demo_drives.csv", index=False)
pd.DataFrame(demo_key).to_csv("model/demo_key.csv", index=False)
print(sorted(os.listdir("model")))
print(pd.DataFrame(demo_key).round(3).to_string(index=False))

## 8. Conclusion (validation, Q1 2026)

- Model: logistic regression on 20 summary features of a 14-day window; the label is "fails within the next 14 days". It was chosen over gradient boosting because the two perform alike, and it is simpler and explainable.
- At 1% false alarms it catches 65.1% of the 129 failed validation drives (range 56 to 73%); at 0.1%, 36.4%; at 0.01%, 9.3%.
- At most 92 of the 129 (71.3%) can be caught from these inputs: 25 have no usable window and 12 look healthy on all four core counters.
- The alarm level depends on the cost ratio R. From R = 20 upward alarms pay off (R = 100: 18.5 false alarms per day, 48 of 129 failed drives caught).
- Limits: 129 failed validation drives; alarm levels were chosen and reported on the same quarter; Q2 2026 has not been opened and is held out for a final test.